# Data Linkage and cleaning

---

First I have standardised the Account_id outputs with 'ACC-XXXXXXXX'

Then, I have removed all exactly identical rows - not rows that just share the same account number in order to avoid removing frequent loanees

Then I have combined all the columns into a large table, which combines all the values from all 3 tables

# Loading the datasets

In [3]:
from pathlib import Path
import pandas as pd


def find_data_directory(start: Path) -> Path:
    """Find the project data folder from the current notebook location."""
    required_files = {
        "applications.csv",
        "performance_12m.csv",
        "experian_consumer.csv",
    }

    for candidate in [start, *start.parents]:
        data_directory = candidate / "data"

        if data_directory.is_dir():
            available_files = {
                path.name for path in data_directory.iterdir()
            }

            if required_files.issubset(available_files):
                return data_directory

    raise FileNotFoundError(
        "Could not find a data folder containing "
        "applications.csv, performance_12m.csv and "
        "experian_consumer.csv."
    )


DATA_DIR = find_data_directory(Path.cwd().resolve())

applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

experian_consumer = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


print(f"Data loaded from: {DATA_DIR}")
print(f"Applications: {applications.shape}")
print(f"Performance:  {performance.shape}")
print(f"Experian:     {experian_consumer.shape}")

Data loaded from: C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data
Applications: (9987, 38)
Performance:  (10012, 13)
Experian:     (10000, 15)


---

# Cleaning and joining the data

In [4]:
import pandas as pd


# Change this to "Account Number" if that is the real column heading.
ACCOUNT_COLUMN = "account_id"


def normalise_account_number(series: pd.Series) -> pd.Series:
    """
    Clean account numbers deterministically.

    Examples:
        ACC-10002060 -> ACC10002060
        acc 10002060 -> ACC10002060
    """
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )

    # Project account numbers must be ACC followed by eight digits.
    return cleaned.where(
        cleaned.str.fullmatch(r"ACC\d{8}", na=False)
    )


# ------------------------------------------------------------------
# 1. Copy the original source tables
# ------------------------------------------------------------------

sources = {
    "applications": applications.copy(),
    "performance": performance.copy(),
    "experian": experian_consumer.copy(),
}


# ------------------------------------------------------------------
# 2. Remove exact duplicate rows from applications only
# ------------------------------------------------------------------

application_rows_before = len(sources["applications"])

sources["applications"] = (
    sources["applications"]
    .drop_duplicates(keep="first")
    .copy()
)

application_rows_after = len(sources["applications"])

deduplication_audit = pd.DataFrame([
    {
        "source": "applications",
        "rows_before": application_rows_before,
        "exact_duplicates_removed": (
            application_rows_before - application_rows_after
        ),
        "rows_after": application_rows_after,
    },
    {
        "source": "performance",
        "rows_before": len(sources["performance"]),
        "exact_duplicates_removed": 0,
        "rows_after": len(sources["performance"]),
    },
    {
        "source": "experian",
        "rows_before": len(sources["experian"]),
        "exact_duplicates_removed": 0,
        "rows_after": len(sources["experian"]),
    },
])


# ------------------------------------------------------------------
# 3. Clean the account number in every table
# ------------------------------------------------------------------

for frame in sources.values():
    frame["account_number_raw"] = frame[ACCOUNT_COLUMN]

    frame["account_number_clean"] = normalise_account_number(
        frame[ACCOUNT_COLUMN]
    )


# ------------------------------------------------------------------
# 4. Identify invalid and non-unique account numbers
# ------------------------------------------------------------------

for frame in sources.values():
    frame["invalid_account_number"] = (
        frame["account_number_clean"].isna()
    )

    frame["duplicate_account_number"] = (
        frame["account_number_clean"].notna()
        & frame["account_number_clean"].duplicated(keep=False)
    )


# Create an audit table before excluding anything.
data_quality_audit = pd.DataFrame([
    {
        "source": name,
        "rows": len(frame),
        "invalid_account_numbers": (
            frame["invalid_account_number"].sum()
        ),
        "duplicate_account_number_rows": (
            frame["duplicate_account_number"].sum()
        ),
        "unique_valid_accounts": (
            frame.loc[
                frame["account_number_clean"].notna(),
                "account_number_clean",
            ].nunique()
        ),
    }
    for name, frame in sources.items()
])


# Preserve excluded records for investigation.
exceptions = pd.concat(
    [
        frame.loc[
            frame["invalid_account_number"]
            | frame["duplicate_account_number"],
            [
                "account_number_raw",
                "account_number_clean",
                "invalid_account_number",
                "duplicate_account_number",
            ],
        ].assign(source=name)
        for name, frame in sources.items()
    ],
    ignore_index=True,
)


# ------------------------------------------------------------------
# 5. Produce one-row-per-account source tables
# ------------------------------------------------------------------

clean = {
    name: frame.loc[
        ~frame["invalid_account_number"]
        & ~frame["duplicate_account_number"]
    ].drop(
        columns=[
            "invalid_account_number",
            "duplicate_account_number",
        ]
    ).copy()
    for name, frame in sources.items()
}


# Verify uniqueness before joining.
for name, frame in clean.items():
    assert frame["account_number_clean"].is_unique, (
        f"{name} does not contain one row per account"
    )


# ------------------------------------------------------------------
# 6. Identify accounts that cannot be linked
# ------------------------------------------------------------------

application_keys = set(
    clean["applications"]["account_number_clean"]
)

performance_keys = set(
    clean["performance"]["account_number_clean"]
)

experian_keys = set(
    clean["experian"]["account_number_clean"]
)


unmatched_accounts = pd.concat(
    [
        pd.DataFrame({
            "account_number_clean": sorted(
                application_keys - performance_keys
            ),
            "issue": "APPLICATION_WITHOUT_PERFORMANCE",
        }),
        pd.DataFrame({
            "account_number_clean": sorted(
                performance_keys - application_keys
            ),
            "issue": "PERFORMANCE_WITHOUT_APPLICATION",
        }),
        pd.DataFrame({
            "account_number_clean": sorted(
                application_keys - experian_keys
            ),
            "issue": "APPLICATION_WITHOUT_EXPERIAN",
        }),
        pd.DataFrame({
            "account_number_clean": sorted(
                experian_keys - application_keys
            ),
            "issue": "EXPERIAN_WITHOUT_APPLICATION",
        }),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------------
# 7. Inner join all three tables
# ------------------------------------------------------------------

large_dataset = (
    clean["applications"]
    .merge(
        clean["performance"].drop(
            columns=[
                ACCOUNT_COLUMN,
                "account_number_raw",
            ]
        ),
        on="account_number_clean",
        how="inner",
        validate="one_to_one",
        suffixes=("_application", "_performance"),
    )
    .merge(
        clean["experian"].drop(
            columns=[
                ACCOUNT_COLUMN,
                "account_number_raw",
            ]
        ),
        on="account_number_clean",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_experian"),
    )
)


# ------------------------------------------------------------------
# 8. Final validation
# ------------------------------------------------------------------

assert large_dataset["account_number_clean"].notna().all()
assert large_dataset["account_number_clean"].is_unique

assert len(large_dataset) == large_dataset[
    "account_number_clean"
].nunique()


linkage_summary = pd.DataFrame([
    {
        "application_rows_after_deduplication":
            len(clean["applications"]),

        "performance_rows_eligible_for_linkage":
            len(clean["performance"]),

        "experian_rows_eligible_for_linkage":
            len(clean["experian"]),

        "fully_linked_accounts":
            len(large_dataset),

        "unmatched_account_references":
            len(unmatched_accounts),
    }
])


display(deduplication_audit)
display(data_quality_audit)
display(linkage_summary)
display(unmatched_accounts.head(20))
display(exceptions.head(20))
display(large_dataset.head())

,source,rows_before,exact_duplicates_removed,rows_after
0,applications,9987,12,9975
1,performance,10012,0,10012
2,experian,10000,0,10000


,source,rows,invalid_account_numbers,duplicate_account_number_rows,unique_valid_accounts
0,applications,9975,0,0,9975
1,performance,10012,62,0,9950
2,experian,10000,0,0,10000


,application_rows_after_deduplication,performance_rows_eligible_for_linkage,experian_rows_eligible_for_linkage,fully_linked_accounts,unmatched_account_references
0,9975,9950,10000,9925,100


,account_number_clean,issue
0,ACC10000133,APPLICATION_WITHOUT_PERFORMANCE
1,ACC10000171,APPLICATION_WITHOUT_PERFORMANCE
2,ACC10000246,APPLICATION_WITHOUT_PERFORMANCE
3,ACC10000468,APPLICATION_WITHOUT_PERFORMANCE
4,ACC10000541,APPLICATION_WITHOUT_PERFORMANCE
5,ACC10000586,APPLICATION_WITHOUT_PERFORMANCE
6,ACC10000956,APPLICATION_WITHOUT_PERFORMANCE
7,ACC10001060,APPLICATION_WITHOUT_PERFORMANCE
8,ACC10001386,APPLICATION_WITHOUT_PERFORMANCE
9,ACC10001486,APPLICATION_WITHOUT_PERFORMANCE


,account_number_raw,account_number_clean,invalid_account_number,duplicate_account_number,source
0,<NA>,<NA>,True,False,performance
1,<NA>,<NA>,True,False,performance
2,ACC-100002XX,<NA>,True,False,performance
3,<NA>,<NA>,True,False,performance
4,<NA>,<NA>,True,False,performance
5,ACC-100005XX,<NA>,True,False,performance
6,ACC-100009XX,<NA>,True,False,performance
7,ACC-100010XX,<NA>,True,False,performance
8,<NA>,<NA>,True,False,performance
9,ACC-100014XX,<NA>,True,False,performance


,account_id,customer_id,customer_name,address_line_1,postcode,date_of_birth,age_at_application,application_date,employment_status,employment_tenure_months,...,E1B07,E1B09,E1B12,E1B13,E1E01,E1E02,E1A01,E1A02,EA1C01,EA4Q01
0,ACC-10000000,CUS-20000000,Aisha Foster,218 Station Lane,NG14 9VP,1981-06-19,42,2024-06-02,part_time,26,...,0,5,1,0,1,1,0,0,N,N
1,ACC-10000001,CUS-20000001,Oliver Clarke,9 Kingfisher Way,LS27 2AP,1993-10-12,29,2023-05-29,retired,100,...,0,3,1,0,2,3,0,0,N,N
2,ACC-10000002,CUS-20000002,Isla Davies,181 Oak Road,CF29 7JL,1992-08-16,30,2023-04-11,self_employed,50,...,0,1,0,0,2,2,0,0,N,N
3,ACC-10000003,CUS-20000003,Maya Roberts,191 Station Lane,B3 8NV,1990-06-04,33,2023-12-27,retired,24,...,1,2,0,0,0,1,0,0,N,N
4,ACC-10000004,CUS-20000004,Priya Khan,205 Kingfisher Way,RG28 6EP,1990-12-07,33,2024-04-08,employed,25,...,1,2,0,0,2,2,0,0,N,T


---

# Code for the excluded population from my modelling dataset - cleaned out of both the In and out sample

In [ ]:
# Unique, valid keys available from each cleaned source.
application_keys = set(
    clean["applications"]["account_number_clean"]
)

performance_keys = set(
    clean["performance"]["account_number_clean"]
)

experian_keys = set(
    clean["experian"]["account_number_clean"]
)


# Accounts must exist uniquely in all three tables.
fully_linked_keys = (
    application_keys
    & performance_keys
    & experian_keys
)


# Applications without a usable performance outcome.
applications_without_performance = (
    clean["applications"]
    .loc[
        ~clean["applications"]["account_number_clean"].isin(
            performance_keys
        )
    ]
    .copy()
)

applications_without_performance["exclusion_reason"] = (
    "NO_PERFORMANCE_REPORT"
)


# Applications without a usable Experian record.
applications_without_experian = (
    clean["applications"]
    .loc[
        ~clean["applications"]["account_number_clean"].isin(
            experian_keys
        )
    ]
    .copy()
)

applications_without_experian["exclusion_reason"] = (
    "NO_EXPERIAN_RECORD"
)


# Performance records that have no corresponding application.
performance_without_application = (
    clean["performance"]
    .loc[
        ~clean["performance"]["account_number_clean"].isin(
            application_keys
        )
    ]
    .copy()
)

performance_without_application["exclusion_reason"] = (
    "NO_APPLICATION_RECORD"
)


# Final one-row-per-account modelling dataset.
model_development_dataset = (
    clean["applications"]
    .loc[
        clean["applications"]["account_number_clean"].isin(
            fully_linked_keys
        )
    ]
    .merge(
        clean["performance"].drop(
            columns=[
                ACCOUNT_COLUMN,
                "account_number_raw",
            ]
        ),
        on="account_number_clean",
        how="inner",
        validate="one_to_one",
        suffixes=("_application", "_performance"),
    )
    .merge(
        clean["experian"].drop(
            columns=[
                ACCOUNT_COLUMN,
                "account_number_raw",
            ]
        ),
        on="account_number_clean",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_experian"),
    )
)


# Validate the final result.
assert model_development_dataset[
    "account_number_clean"
].notna().all()

assert model_development_dataset[
    "account_number_clean"
].is_unique

assert len(model_development_dataset) == len(fully_linked_keys)